# Customer Requirement Analysis
## 01 - Data Collection

### Objective

This notebook documents the collection and initial inspection
of user-generated customer feedback datasets for the research
project.

### Research pipeline

Data Sources
→ Data Collection
→ Data Validation
→ Data Cleaning
→ Requirement Extraction
→ Semantic Clustering
→ Requirement Prioritization

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERMEDIATE_DATA_DIR = PROJECT_ROOT / "data" / "intermediate"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Intermediate data:", INTERMEDIATE_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Metadata:", METADATA_DIR)

Project root: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis
Raw data: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\raw
Intermediate data: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\intermediate
Processed data: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\processed
Metadata: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\metadata


In [3]:
import pandas as pd
import numpy as np
import openpyxl
import ipykernel

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("OpenPyXL:", openpyxl.__version__)
print("Environment: OK")

Pandas: 3.0.6
NumPy: 2.5.3
OpenPyXL: 3.1.5
Environment: OK


In [4]:
# Dataset configuration

RESTAURANT_XML = RAW_DATA_DIR / "semeval2015" / "ABSA-15_Restaurants_Train_Final.xml"
LAPTOP_XML = RAW_DATA_DIR / "semeval2015" / "ABSA-15_Laptops_Train_Data.xml"

print("Restaurant dataset:", RESTAURANT_XML)
print("Laptop dataset:", LAPTOP_XML)

print("\nRestaurant file exists:", RESTAURANT_XML.exists())
print("Laptop file exists:", LAPTOP_XML.exists())

if RESTAURANT_XML.exists():
    print("Restaurant file size:", RESTAURANT_XML.stat().st_size, "bytes")

if LAPTOP_XML.exists():
    print("Laptop file size:", LAPTOP_XML.stat().st_size, "bytes")

Restaurant dataset: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\raw\semeval2015\ABSA-15_Restaurants_Train_Final.xml
Laptop dataset: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\raw\semeval2015\ABSA-15_Laptops_Train_Data.xml

Restaurant file exists: True
Laptop file exists: True
Restaurant file size: 476793 bytes
Laptop file size: 543091 bytes


In [5]:
# Inspect the raw XML structure

import xml.etree.ElementTree as ET

restaurant_tree = ET.parse(RESTAURANT_XML)
restaurant_root = restaurant_tree.getroot()

laptop_tree = ET.parse(LAPTOP_XML)
laptop_root = laptop_tree.getroot()

print("Restaurant root tag:", restaurant_root.tag)
print("Laptop root tag:", laptop_root.tag)

print("\nRestaurant first-level elements:")
for child in restaurant_root:
    print(" -", child.tag)

print("\nLaptop first-level elements:")
for child in laptop_root:
    print(" -", child.tag)

Restaurant root tag: Reviews
Laptop root tag: Reviews

Restaurant first-level elements:
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 - Review
 -

In [6]:
# Inspect one complete review from each dataset

restaurant_review = restaurant_root.find("Review")
laptop_review = laptop_root.find("Review")

print("===== RESTAURANT REVIEW =====")
print(ET.tostring(restaurant_review, encoding="unicode"))

print("\n===== LAPTOP REVIEW =====")
print(ET.tostring(laptop_review, encoding="unicode"))

===== RESTAURANT REVIEW =====
<Review rid="1004293">
        <sentences>
            <sentence id="1004293:0">
                <text>Judging from previous posts this used to be a good place, but not any longer.</text>
                <Opinions>
                    <Opinion target="place" category="RESTAURANT#GENERAL" polarity="negative" from="51" to="56" />
                </Opinions>
            </sentence>
            <sentence id="1004293:1">
                <text>We, there were four of us, arrived at noon - the place was empty - and the staff acted like we were imposing on them and they were very rude.</text>
                <Opinions>
                    <Opinion target="staff" category="SERVICE#GENERAL" polarity="negative" from="75" to="80" />
                </Opinions>
            </sentence>
            <sentence id="1004293:2">
                <text>They never brought us complimentary noodles, ignored repeated requests for sugar, and threw our dishes on the table.</text>
    

In [7]:
# Parse SemEval reviews into structured records

def parse_semeval_xml(xml_path, domain):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    records = []

    for review in root.findall("Review"):
        review_id = review.get("rid")

        sentences = review.find("sentences")

        if sentences is None:
            continue

        for sentence in sentences.findall("sentence"):
            sentence_id = sentence.get("id")
            text_element = sentence.find("text")

            if text_element is None or text_element.text is None:
                review_text = ""
            else:
                review_text = text_element.text.strip()

            opinions = sentence.find("Opinions")

            if opinions is None:
                records.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "domain": domain,
                    "review_text": review_text,
                    "target": None,
                    "category": None,
                    "polarity": None
                })
                continue

            opinion_elements = opinions.findall("Opinion")

            if not opinion_elements:
                records.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "domain": domain,
                    "review_text": review_text,
                    "target": None,
                    "category": None,
                    "polarity": None
                })
                continue

            for opinion in opinion_elements:
                records.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "domain": domain,
                    "review_text": review_text,
                    "target": opinion.get("target"),
                    "category": opinion.get("category"),
                    "polarity": opinion.get("polarity")
                })

    return pd.DataFrame(records)

In [8]:
# Parse both datasets

restaurant_df = parse_semeval_xml(
    RESTAURANT_XML,
    domain="restaurant"
)

laptop_df = parse_semeval_xml(
    LAPTOP_XML,
    domain="laptop"
)

print("Restaurant records:", len(restaurant_df))
print("Laptop records:", len(laptop_df))

print("\nRestaurant columns:")
print(restaurant_df.columns.tolist())

print("\nLaptop columns:")
print(laptop_df.columns.tolist())

Restaurant records: 1849
Laptop records: 2314

Restaurant columns:
['review_id', 'sentence_id', 'domain', 'review_text', 'target', 'category', 'polarity']

Laptop columns:
['review_id', 'sentence_id', 'domain', 'review_text', 'target', 'category', 'polarity']


In [9]:
# Use a precise name for the text extracted from each sentence

restaurant_df = restaurant_df.rename(
    columns={"review_text": "sentence_text"}
)

laptop_df = laptop_df.rename(
    columns={"review_text": "sentence_text"}
)

print("Restaurant columns:")
print(restaurant_df.columns.tolist())

print("\nLaptop columns:")
print(laptop_df.columns.tolist())

Restaurant columns:
['review_id', 'sentence_id', 'domain', 'sentence_text', 'target', 'category', 'polarity']

Laptop columns:
['review_id', 'sentence_id', 'domain', 'sentence_text', 'target', 'category', 'polarity']


In [10]:
# Basic data-quality audit

combined_df = pd.concat(
    [restaurant_df, laptop_df],
    ignore_index=True
)

print("Total records:", len(combined_df))

print("\nRecords by domain:")
print(combined_df["domain"].value_counts())

print("\nPolarity distribution:")
print(combined_df["polarity"].value_counts(dropna=False))

print("\nMissing values:")
print(combined_df.isna().sum())

print("\nEmpty sentence texts:")
print((combined_df["sentence_text"].str.strip() == "").sum())

print("\nUnique review IDs:")
print(combined_df["review_id"].nunique())

print("\nUnique sentence IDs:")
print(combined_df["sentence_id"].nunique())

Total records: 4163

Records by domain:
domain
laptop        2314
restaurant    1849
Name: count, dtype: int64

Polarity distribution:
polarity
positive    2301
negative    1168
NaN          535
neutral      159
Name: count, dtype: int64

Missing values:
review_id           0
sentence_id         0
domain              0
sentence_text       0
target           2509
category          535
polarity          535
dtype: int64

Empty sentence texts:
0

Unique review IDs:
531

Unique sentence IDs:
3054


In [11]:
# Investigate missing and special annotation values

print("===== TARGET VALUES =====")
print(combined_df["target"].value_counts(dropna=False).head(20))

print("\n===== CATEGORY VALUES =====")
print(combined_df["category"].value_counts(dropna=False).head(20))

print("\n===== POLARITY BY DOMAIN =====")
print(
    pd.crosstab(
        combined_df["domain"],
        combined_df["polarity"],
        dropna=False
    )
)

print("\n===== LITERAL 'NULL' TARGETS =====")
print(
    (combined_df["target"] == "NULL").sum()
)

print("\n===== ACTUAL MISSING TARGETS =====")
print(
    combined_df["target"].isna().sum()
)

print("\n===== RECORDS WITH NO OPINION ANNOTATION =====")
no_opinion = combined_df["polarity"].isna()
print(no_opinion.sum())

print("\nExamples of records without opinion annotations:")
display(
    combined_df.loc[
        no_opinion,
        ["review_id", "sentence_id", "domain", "sentence_text"]
    ].head(10)
)

===== TARGET VALUES =====
target
None          2314
NULL           375
NaN            195
food           146
service         96
place           80
restaurant      29
staff           26
Service         21
pizza           20
atmosphere      20
sushi           19
decor           14
Food            12
menu            12
ambience        11
dishes          10
portions         9
waiter           9
bagels           9
Name: count, dtype: int64

===== CATEGORY VALUES =====
category
FOOD#QUALITY                     581
NaN                              535
LAPTOP#GENERAL                   413
RESTAURANT#GENERAL               269
SERVICE#GENERAL                  268
AMBIENCE#GENERAL                 183
LAPTOP#DESIGN_FEATURES           162
LAPTOP#OPERATION_PERFORMANCE     157
LAPTOP#QUALITY                   152
SUPPORT#QUALITY                  129
LAPTOP#USABILITY                 104
FOOD#STYLE_OPTIONS                93
LAPTOP#MISCELLANEOUS              91
LAPTOP#PRICE                      76
COMPA

,review_id,sentence_id,domain,sentence_text
34,1041457,1041457:0,restaurant,"I had my eyes on this place, promising myself ..."
42,1041457,1041457:5,restaurant,"Anyways, if you're in the neighborhood to eat ..."
46,1053543,1053543:0,restaurant,We ate outside at Haru's Sake bar because Haru...
47,1053543,1053543:1,restaurant,What's the difference between the two?
59,1058221,1058221:0,restaurant,"i have eaten here a handful of times, for no r..."
60,1058221,1058221:1,restaurant,"(i hang out, and live, in the neighborhood..)"
78,1074868,1074868:0,restaurant,I went to this restaurant with a woman that I ...
79,1074868,1074868:1,restaurant,She lives nearby but had never gone to this es...
85,1084394,1084394:0,restaurant,"Having hunted around for a quiet, romantic, ye..."
87,1084394,1084394:2,restaurant,"The service was attentive, yet unimposing, the..."


In [12]:
# Check whether sentence IDs are repeated because of multiple opinions

sentence_counts = combined_df["sentence_id"].value_counts()

print("Sentences represented by more than one record:")
print((sentence_counts > 1).sum())

print("\nMaximum records for one sentence:")
print(sentence_counts.max())

print("\nExample of a sentence with multiple records:")
example_sentence_id = sentence_counts[sentence_counts > 1].index[0]

display(
    combined_df[
        combined_df["sentence_id"] == example_sentence_id
    ]
)

Sentences represented by more than one record:
824

Maximum records for one sentence:
8

Example of a sentence with multiple records:


,review_id,sentence_id,domain,sentence_text,target,category,polarity
1100,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,sushi,FOOD#QUALITY,positive
1101,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,ceviche mix (special),FOOD#QUALITY,positive
1102,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,crab dumplings,FOOD#QUALITY,positive
1103,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,assorted sashimi,FOOD#QUALITY,positive
1104,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,sushi,FOOD#QUALITY,positive
1105,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,rolls,FOOD#QUALITY,positive
1106,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,two types of sake,DRINKS#QUALITY,positive
1107,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,banana tempura,FOOD#QUALITY,positive


In [13]:
# Create domain-aware annotation status

combined_df["has_opinion_annotation"] = combined_df["polarity"].notna()

combined_df["has_aspect_category"] = combined_df["category"].notna()

combined_df["has_target"] = (
    combined_df["domain"].eq("restaurant")
    & combined_df["target"].notna()
    & combined_df["target"].ne("NULL")
)

print("Opinion annotation status:")
print(combined_df["has_opinion_annotation"].value_counts())

print("\nAspect category status:")
print(combined_df["has_aspect_category"].value_counts())

print("\nExplicit restaurant target status:")
print(combined_df["has_target"].value_counts())

print("\nBy domain:")
display(
    combined_df.groupby("domain")[
        ["has_opinion_annotation",
         "has_aspect_category",
         "has_target"]
    ].sum()
)

Opinion annotation status:
has_opinion_annotation
True     3628
False     535
Name: count, dtype: int64

Aspect category status:
has_aspect_category
True     3628
False     535
Name: count, dtype: int64

Explicit restaurant target status:
has_target
False    2884
True     1279
Name: count, dtype: int64

By domain:


,has_opinion_annotation,has_aspect_category,has_target
domain,,,
laptop,1974,1974,0
restaurant,1654,1654,1279


In [14]:
# Separate annotated and non-annotated records

annotated_df = combined_df[
    combined_df["has_opinion_annotation"]
].copy()

unannotated_df = combined_df[
    ~combined_df["has_opinion_annotation"]
].copy()

print("Annotated records:", len(annotated_df))
print("Non-annotated records:", len(unannotated_df))

print("\nAnnotated records by domain:")
print(annotated_df["domain"].value_counts())

print("\nNon-annotated records by domain:")
print(unannotated_df["domain"].value_counts())

Annotated records: 3628
Non-annotated records: 535

Annotated records by domain:
domain
laptop        1974
restaurant    1654
Name: count, dtype: int64

Non-annotated records by domain:
domain
laptop        340
restaurant    195
Name: count, dtype: int64


In [15]:
# Check for exact duplicate annotation records

duplicate_mask = combined_df.duplicated(
    subset=[
        "review_id",
        "sentence_id",
        "domain",
        "sentence_text",
        "target",
        "category",
        "polarity"
    ],
    keep=False
)

print("Exact duplicate records:", duplicate_mask.sum())

if duplicate_mask.any():
    display(
        combined_df.loc[duplicate_mask]
        .sort_values(["review_id", "sentence_id"])
        .head(20)
    )
else:
    print("No exact duplicate records found.")

Exact duplicate records: 4


,review_id,sentence_id,domain,sentence_text,target,category,polarity,has_opinion_annotation,has_aspect_category,has_target
512,1349391,1349391:1,restaurant,"sometimes i get bad food and bad service, some...",service,SERVICE#GENERAL,negative,True,True,True
513,1349391,1349391:1,restaurant,"sometimes i get bad food and bad service, some...",service,SERVICE#GENERAL,negative,True,True,True
1100,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,sushi,FOOD#QUALITY,positive,True,True,True
1104,1726427,1726427:1,restaurant,We are very particular about sushi and were bo...,sushi,FOOD#QUALITY,positive,True,True,True


In [16]:
# Text quality audit

print("===== TEXT QUALITY AUDIT =====")

empty_text = combined_df["sentence_text"].str.strip() == ""
whitespace_only = combined_df["sentence_text"].str.fullmatch(r"\s*")

print("Empty text records:", empty_text.sum())
print("Whitespace-only records:", whitespace_only.sum())

print("\n===== TEXT LENGTH STATISTICS =====")
print(combined_df["sentence_text"].str.len().describe())

print("\n===== SHORTEST SENTENCES =====")
display(
    combined_df.loc[
        combined_df["sentence_text"].str.len().nsmallest(10).index,
        ["review_id", "sentence_id", "domain", "sentence_text"]
    ]
)

===== TEXT QUALITY AUDIT =====
Empty text records: 0
Whitespace-only records: 0

===== TEXT LENGTH STATISTICS =====
count    4163.000000
mean       77.142686
std        46.244818
min         2.000000
25%        45.000000
50%        68.000000
75%        99.000000
max       400.000000
Name: sentence_text, dtype: float64

===== SHORTEST SENTENCES =====


,review_id,sentence_id,domain,sentence_text
839,1573534,1573534:8,restaurant,10
615,1413697,1413697:6,restaurant,LOL
935,1642666,1642666:2,restaurant,Why?
3488,189,189:0,laptop,Shiny
1546,561054,561054:4,restaurant,Ahhh...
1142,1730127,1730127:3,restaurant,Go here.
1653,680345,680345:3,restaurant,Amazing!
2187,247,247:4,laptop,Love it.
2491,143,143:0,laptop,Runs Hot
4105,128,128:11,laptop,"Oh, boy!"


In [17]:
# Validate polarity labels

valid_polarities = {"positive", "negative", "neutral"}

invalid_polarity = combined_df.loc[
    combined_df["polarity"].notna()
    & ~combined_df["polarity"].isin(valid_polarities)
]

print("Valid polarity records:",
      combined_df["polarity"].isin(valid_polarities).sum())

print("Invalid polarity records:",
      len(invalid_polarity))

if len(invalid_polarity) > 0:
    display(
        invalid_polarity[
            ["review_id", "sentence_id", "domain", "sentence_text", "polarity"]
        ].head(20)
    )
else:
    print("All non-missing polarity labels are valid.")

Valid polarity records: 3628
Invalid polarity records: 0
All non-missing polarity labels are valid.


In [18]:
# Validate aspect category format

annotated_categories = combined_df["category"].dropna().astype(str)

invalid_category = annotated_categories[
    ~annotated_categories.str.contains("#", regex=False)
]

print("Total non-missing categories:", len(annotated_categories))
print("Valid category format:", len(annotated_categories) - len(invalid_category))
print("Invalid category format:", len(invalid_category))

if len(invalid_category) > 0:
    print("\nInvalid category values:")
    print(invalid_category.value_counts())
else:
    print("All non-missing categories follow the expected format.")

Total non-missing categories: 3628
Valid category format: 3628
Invalid category format: 0
All non-missing categories follow the expected format.


In [19]:
# Inspect aspect categories by domain

print("===== RESTAURANT CATEGORIES =====")
restaurant_categories = (
    combined_df.loc[
        combined_df["domain"] == "restaurant",
        "category"
    ]
    .dropna()
    .value_counts()
)

display(restaurant_categories)

print("\n===== LAPTOP CATEGORIES =====")
laptop_categories = (
    combined_df.loc[
        combined_df["domain"] == "laptop",
        "category"
    ]
    .dropna()
    .value_counts()
)

display(laptop_categories)

===== RESTAURANT CATEGORIES =====


category
FOOD#QUALITY                581
RESTAURANT#GENERAL          269
SERVICE#GENERAL             268
AMBIENCE#GENERAL            183
FOOD#STYLE_OPTIONS           93
RESTAURANT#MISCELLANEOUS     62
FOOD#PRICES                  54
RESTAURANT#PRICES            48
DRINKS#QUALITY               34
DRINKS#STYLE_OPTIONS         26
LOCATION#GENERAL             20
DRINKS#PRICES                15
FOOD#GENERAL                  1
Name: count, dtype: int64


===== LAPTOP CATEGORIES =====


category
LAPTOP#GENERAL                        413
LAPTOP#DESIGN_FEATURES                162
LAPTOP#OPERATION_PERFORMANCE          157
LAPTOP#QUALITY                        152
SUPPORT#QUALITY                       129
                                     ... 
FANS_COOLING#DESIGN_FEATURES            1
MULTIMEDIA_DEVICES#MISCELLANEOUS        1
POWER_SUPPLY#MISCELLANEOUS              1
PORTS#OPERATION_PERFORMANCE             1
FANS_COOLING#OPERATION_PERFORMANCE      1
Name: count, Length: 81, dtype: int64

In [20]:
# Check category-domain consistency

category_prefix = (
    combined_df["category"]
    .dropna()
    .astype(str)
    .str.split("#")
    .str[0]
)

category_domain_df = combined_df.loc[
    combined_df["category"].notna(),
    ["domain", "category"]
].copy()

category_domain_df["category_prefix"] = (
    category_domain_df["category"]
    .astype(str)
    .str.split("#")
    .str[0]
)

print("===== CATEGORY PREFIXES BY DOMAIN =====")

display(
    pd.crosstab(
        category_domain_df["domain"],
        category_domain_df["category_prefix"]
    )
)

===== CATEGORY PREFIXES BY DOMAIN =====


category_prefix,AMBIENCE,BATTERY,COMPANY,CPU,DISPLAY,DRINKS,FANS_COOLING,FOOD,GRAPHICS,HARDWARE,...,OPTICAL_DRIVES,OS,PORTS,POWER_SUPPLY,RESTAURANT,SERVICE,SHIPPING,SOFTWARE,SUPPORT,WARRANTY
domain,,,,,,,,,,,,,,,,,,,,,
laptop,0,77,69,11,78,0,4,0,24,3,...,2,42,6,18,0,0,11,71,142,6
restaurant,183,0,0,0,0,75,0,729,0,0,...,0,0,0,0,379,268,0,0,0,0


In [21]:
# Create cleaned dataset
# Only exact duplicate annotation records are removed.

cleaned_df = combined_df.copy()

before_count = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates(
    subset=[
        "review_id",
        "sentence_id",
        "domain",
        "sentence_text",
        "target",
        "category",
        "polarity"
    ],
    keep="first"
).reset_index(drop=True)

after_count = len(cleaned_df)

print("===== DUPLICATE REMOVAL =====")
print("Records before cleaning:", before_count)
print("Records after cleaning:", after_count)
print("Exact duplicates removed:", before_count - after_count)

===== DUPLICATE REMOVAL =====
Records before cleaning: 4163
Records after cleaning: 4161
Exact duplicates removed: 2


In [22]:
# Cleaning summary

print("===== DATA CLEANING SUMMARY =====")

print("Original records:", len(combined_df))
print("Cleaned records:", len(cleaned_df))
print("Redundant duplicate rows removed:",
      len(combined_df) - len(cleaned_df))

print("\nOriginal records by domain:")
print(combined_df["domain"].value_counts())

print("\nCleaned records by domain:")
print(cleaned_df["domain"].value_counts())

print("\nMissing values after cleaning:")
print(cleaned_df.isna().sum())

===== DATA CLEANING SUMMARY =====
Original records: 4163
Cleaned records: 4161
Redundant duplicate rows removed: 2

Original records by domain:
domain
laptop        2314
restaurant    1849
Name: count, dtype: int64

Cleaned records by domain:
domain
laptop        2314
restaurant    1847
Name: count, dtype: int64

Missing values after cleaning:
review_id                    0
sentence_id                  0
domain                       0
sentence_text                0
target                    2509
category                   535
polarity                   535
has_opinion_annotation       0
has_aspect_category          0
has_target                   0
dtype: int64


In [23]:
# Save cleaned SemEval dataset to intermediate data

INTERMEDIATE_SEMEVAL_DIR = INTERMEDIATE_DATA_DIR / "semeval2015"
INTERMEDIATE_SEMEVAL_DIR.mkdir(parents=True, exist_ok=True)

INTERMEDIATE_CSV = (
    INTERMEDIATE_SEMEVAL_DIR / "semeval2015_combined_cleaned.csv"
)

cleaned_df.to_csv(INTERMEDIATE_CSV, index=False)

print("Cleaned dataset saved successfully.")
print("Path:", INTERMEDIATE_CSV)
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

Cleaned dataset saved successfully.
Path: c:\Users\roshn\Desktop\Customer Analysis\customer-requirement-analysis\data\intermediate\semeval2015\semeval2015_combined_cleaned.csv
Rows: 4161
Columns: 10


In [24]:
# Verify the saved intermediate dataset

verified_df = pd.read_csv(INTERMEDIATE_CSV)

print("===== SAVED DATASET VERIFICATION =====")
print("Rows:", len(verified_df))
print("Columns:", len(verified_df.columns))

print("\nColumn names:")
print(list(verified_df.columns))

print("\nDomain distribution:")
print(verified_df["domain"].value_counts())

print("\nFirst 5 records:")
display(verified_df.head())

===== SAVED DATASET VERIFICATION =====
Rows: 4161
Columns: 10

Column names:
['review_id', 'sentence_id', 'domain', 'sentence_text', 'target', 'category', 'polarity', 'has_opinion_annotation', 'has_aspect_category', 'has_target']

Domain distribution:
domain
laptop        2314
restaurant    1847
Name: count, dtype: int64

First 5 records:


,review_id,sentence_id,domain,sentence_text,target,category,polarity,has_opinion_annotation,has_aspect_category,has_target
0,1004293,1004293:0,restaurant,Judging from previous posts this used to be a ...,place,RESTAURANT#GENERAL,negative,True,True,True
1,1004293,1004293:1,restaurant,"We, there were four of us, arrived at noon - t...",staff,SERVICE#GENERAL,negative,True,True,True
2,1004293,1004293:2,restaurant,"They never brought us complimentary noodles, i...",NaN,SERVICE#GENERAL,negative,True,True,False
3,1004293,1004293:3,restaurant,The food was lousy - too sweet or too salty an...,food,FOOD#QUALITY,negative,True,True,True
4,1004293,1004293:3,restaurant,The food was lousy - too sweet or too salty an...,portions,FOOD#STYLE_OPTIONS,negative,True,True,True


## Data Cleaning Report

### Dataset
SemEval-2015 Task 12 was used as the primary benchmark dataset, combining
restaurant and laptop review domains.

### Initial Dataset
- Total parsed records: 4,163
- Laptop records: 2,314
- Restaurant records: 1,849

### Data Quality Checks

The following validation checks were performed:

1. **Missing text validation**
   - Empty sentence records: 0
   - Whitespace-only records: 0

2. **Polarity validation**
   - Valid polarity records: 3,628
   - Invalid polarity records: 0
   - Valid labels: positive, negative, neutral

3. **Aspect category validation**
   - Non-missing category records: 3,628
   - Invalid category formats: 0
   - Expected format: ENTITY#ATTRIBUTE

4. **Annotation validation**
   - Annotated records: 3,628
   - Non-annotated records: 535
   - Non-annotated records were retained because they contain valid review
     sentences without opinion annotations.

5. **Duplicate validation**
   - Records involved in exact duplicate groups: 4
   - Redundant duplicate rows removed: 2

### Cleaning Policy

Only exact duplicate annotation records were removed.

Missing target, category, and polarity values were not automatically removed
because their presence depends on the annotation structure of the dataset.
In particular, target annotations are domain-dependent, while sentences
without opinion annotations remain useful for the dataset audit and broader
UGC processing pipeline.

### Final Intermediate Dataset

- Final records: 4,161
- Laptop records: 2,314
- Restaurant records: 1,847

The cleaned dataset was saved as:

`data/intermediate/semeval2015/semeval2015_combined_cleaned.csv`

## Data Collection and Cleaning Complete

The SemEval-2015 restaurant and laptop datasets were parsed, combined,
audited, and cleaned.

Final intermediate dataset:
- Records: 4,161
- Laptop: 2,314
- Restaurant: 1,847
- Redundant duplicate rows removed: 2
- Empty text records: 0
- Invalid polarity records: 0
- Invalid category-format records: 0

The cleaned dataset is generated locally under:

`data/intermediate/semeval2015/semeval2015_combined_cleaned.csv`

The intermediate dataset is excluded from Git because it is derived from
a benchmark dataset. The notebook provides the reproducible processing
pipeline.